# Smart Patio Shield - ConvLSTM and the diurnal control (Notebook 08)

Two questions raised in supervision, tested in one session.

**Q1 - Does the architecture matter, or was the sampling interval the problem?**
Notebook 07 stacked consecutive hourly frames as extra channels and performance
declined (test PR-AUC 0.4034 -> 0.3809 -> 0.3638 for n = 1, 2, 3). Channel stacking
throws away temporal order: a convolution over stacked channels cannot tell that one
frame precedes another. A ConvLSTM keeps the spatial map and carries a recurrent state
across frames, so it is a genuinely different test rather than a rerun.

**Q2 - Is any apparent motion signal really a time-of-day signal?**
Rainfall here has a strong diurnal cycle (roughly 1.8% positive at 06:00 against 31.1%
at 14:00) and the pressure record shows the semidiurnal atmospheric tide. A model can
score well by learning "it is mid-afternoon" without learning anything about motion.
The control: give each frame its own diurnal (24 h) and semidiurnal (12 h) harmonics as
explicit input planes. Any remaining benefit from recurrence must then be information
beyond timing.

**Expectations, stated before running.** The earlier negative may be caused by the
hourly interval rather than the architecture: at trade-wind speeds a cell can cross the
~128 km patch inside an hour, so consecutive frames approximate independent snapshots.
If that is the binding constraint, ConvLSTM will not rescue it. The ConvLSTM is also
trained from scratch while the ResNet-18 is fine-tuned from ImageNet, so it starts at a
disadvantage. A null result here strengthens the sub-hourly-imagery argument rather than
weakening it.

Sections
- **0** Session setup
- **1** Datasets, channel-order verification, time-harmonic planes
- **2** Train ConvLSTM, with and without the diurnal control
- **3** Stratified evaluation by time of day
- **4** Fusion with the ConvLSTM vision branch
- **5** Save results

## Section 0 - Session setup
Same idempotent setup as notebooks 05 and 07. Requires `src/models/goes_temporal.py`
and `src/models/convlstm.py` to be committed and pushed.

In [ ]:
import os, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE - Runtime > Change runtime type > T4 GPU")
if not os.path.exists("/content/smart-patio-shield"):
    from getpass import getpass
    tok = getpass("GitHub token: ")
    !git clone https://{tok}@github.com/romayneg/smart-patio-shield.git /content/smart-patio-shield
%cd /content/smart-patio-shield
!git pull -q
print("code up to date")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = "/content/drive/MyDrive/smart-patio-shield"

import shutil
def relink(target, linkname):
    if os.path.islink(linkname):   os.unlink(linkname)
    elif os.path.isdir(linkname):  shutil.rmtree(linkname)
    elif os.path.exists(linkname): os.remove(linkname)
    parent = os.path.dirname(linkname)
    if parent: os.makedirs(parent, exist_ok=True)
    os.symlink(target, linkname)

relink(f"{DRIVE}/data/goes",                   "data/raw/goes")
relink(f"{DRIVE}/data/image_labels.parquet",   "data/processed/image_labels.parquet")
relink(f"{DRIVE}/data/patio_features.parquet", "data/processed/patio_features.parquet")
relink(f"{DRIVE}/models",                      "models")
print("day-files:", len(os.listdir("data/raw/goes")))
for f in ["src/models/goes_temporal.py", "src/models/convlstm.py"]:
    assert os.path.exists(f), f"{f} not found - commit & push it, then re-run Section 0."
print("setup complete")

## Section 1 - Datasets, channel order, and time-harmonic planes

Same fairness control as notebook 07: the n = 3 keys define the common sample set and
every configuration is restricted to those identical hours, so nothing is compared
against an easier sample.

The channel-order check below matters. `ConvLSTMClassifier` reshapes `(C*T, H, W)` into
`(T, C, H, W)`, which is only correct if frames sit in contiguous blocks, oldest first.
Rather than trust that, the cell verifies it: with shared normalisation statistics, the
final frame-block of an n = 3 tensor must equal the n = 1 tensor for the same hour. It
fails loudly if the layout differs.

In [ ]:
import importlib, numpy as np, torch
import src.models.goes_temporal as gt
import src.models.goes_dataset as gd
import src.models.convlstm as cl
for m in (gt, gd, cl): importlib.reload(m)

patches = gd._load_all_patches()
print(f"patches in memory: {len(patches):,}")

BANDS = gt.IR_ONLY      # isolate motion from the C02 day/night confound
CPF   = len(BANDS)      # channels per frame
NF    = 3               # frames per sample

def build(split, n_frames, norm=None, restrict_keys=None):
    ds = gt.GoesTemporalDataset(split, channels=BANDS, n_frames=n_frames,
                                norm_stats=norm, patches=patches)
    if restrict_keys is not None:
        keep = set(restrict_keys)
        ds.keys = [k for k in ds.keys if k in keep]
    return ds

common = {}
for split in ["train", "val", "test"]:
    common[split] = set(build(split, NF).keys)
    print(f"{split}: common (n={NF}-eligible) samples = {len(common[split]):,}")

In [ ]:
# Verify frame-major channel ordering before trusting the reshape.
probe3 = build("val", NF, restrict_keys=common["val"])
NORM_PROBE = probe3.norm_stats
probe1 = build("val", 1, norm=NORM_PROBE, restrict_keys=common["val"])

k = probe3.keys[0]
x3 = probe3[probe3.keys.index(k)][0].numpy()
x1 = probe1[probe1.keys.index(k)][0].numpy()
print("n=3 tensor:", x3.shape, " n=1 tensor:", x1.shape)

last_block = x3[-CPF:]
if np.allclose(last_block, x1, atol=1e-5):
    print("PASS - frames are contiguous blocks, oldest first; the reshape is correct.")
else:
    diffs = [float(np.abs(x3[i*CPF:(i+1)*CPF] - x1).mean()) for i in range(NF)]
    print("FAIL - last block does not match the single-frame tensor.")
    print("mean abs diff by block:", [round(d, 5) for d in diffs])
    raise AssertionError("Channel layout differs from the frame-major assumption. "
                         "Check goes_temporal.py before training.")

In [ ]:
from torch.utils.data import Dataset
from datetime import datetime

class TimeHarmonics(Dataset):
    """Append per-frame diurnal (24 h) and semidiurnal (12 h) harmonics.

    Each frame receives the harmonics for its own hour, so frame t-2 is labelled two
    hours earlier than the current frame. With timing supplied explicitly, any gain
    from recurrence has to be information the clock does not already provide.
    """

    def __init__(self, base, n_frames, cpf):
        self.base, self.n_frames, self.cpf = base, n_frames, cpf
        self.keys, self.label_of = base.keys, base.label_of
        self.extra = 4

    def __len__(self):
        return len(self.base)

    @staticmethod
    def _hour(key):
        return datetime.fromisoformat(key.split("|", 1)[1]).hour

    def __getitem__(self, i):
        x, y = self.base[i]
        h0 = self._hour(self.keys[i])
        hgt, wid = x.shape[-2:]
        blocks = []
        for j in range(self.n_frames):
            h = (h0 - (self.n_frames - 1 - j)) % 24
            vals = [np.sin(2*np.pi*h/24), np.cos(2*np.pi*h/24),
                    np.sin(4*np.pi*h/24), np.cos(4*np.pi*h/24)]
            planes = torch.stack([torch.full((hgt, wid), float(v)) for v in vals])
            blocks.append(torch.cat([x[j*self.cpf:(j+1)*self.cpf], planes], dim=0))
        return torch.cat(blocks, dim=0), y

print("TimeHarmonics ready: adds 4 planes per frame (24 h and 12 h sin/cos).")

## Section 2 - Train the ConvLSTM, with and without the diurnal control

Two configurations, identical apart from the timing planes:

- **A - ConvLSTM (plain)**: 2 IR channels per frame, 3 frames.
- **B - ConvLSTM + time harmonics**: 6 channels per frame (2 IR + 4 timing).

Training recipe matches `cnn.train_model`: class-weighted BCE, Adam, PR-AUC early
stopping. Learning rate is 1e-3 rather than 1e-4 because this network is trained from
scratch instead of fine-tuned.

In [ ]:
from torch.utils.data import DataLoader

train_base = build("train", NF, restrict_keys=common["train"])
NORM = train_base.norm_stats
val_base  = build("val",  NF, norm=NORM, restrict_keys=common["val"])
test_base = build("test", NF, norm=NORM, restrict_keys=common["test"])
print(f"samples train/val/test: {len(train_base)}/{len(val_base)}/{len(test_base)}")

results = {}

print("\n" + "="*30 + "  A: ConvLSTM (plain)  " + "="*30)
model_a, hist_a = cl.train_convlstm(train_base, val_base, channels_per_frame=CPF,
                                    n_frames=NF, hidden=64, epochs=25,
                                    batch_size=64, lr=1e-3, patience=4)
dev = "cuda" if torch.cuda.is_available() else "cpu"
ev_a = cl.evaluate(model_a, DataLoader(test_base, batch_size=128, num_workers=2), dev)
results["convlstm_plain"] = {"val_pr_auc": round(max(h["val_pr_auc"] for h in hist_a), 4),
                             "test_pr_auc": round(ev_a["pr_auc"], 4),
                             "channels_per_frame": CPF, "n_frames": NF}
print(f"A test PR-AUC: {ev_a['pr_auc']:.4f}")

In [ ]:
print("\n" + "="*26 + "  B: ConvLSTM + time harmonics  " + "="*26)
train_t = TimeHarmonics(train_base, NF, CPF)
val_t   = TimeHarmonics(val_base,   NF, CPF)
test_t  = TimeHarmonics(test_base,  NF, CPF)

model_b, hist_b = cl.train_convlstm(train_t, val_t, channels_per_frame=CPF + 4,
                                    n_frames=NF, hidden=64, epochs=25,
                                    batch_size=64, lr=1e-3, patience=4)
ev_b = cl.evaluate(model_b, DataLoader(test_t, batch_size=128, num_workers=2), dev)
results["convlstm_time_controlled"] = {
    "val_pr_auc": round(max(h["val_pr_auc"] for h in hist_b), 4),
    "test_pr_auc": round(ev_b["pr_auc"], 4),
    "channels_per_frame": CPF + 4, "n_frames": NF}
print(f"B test PR-AUC: {ev_b['pr_auc']:.4f}")

print("\n==== ConvLSTM summary (shared samples, IR-only) ====")
print(f"{'config':<28} {'val':>8} {'test':>8}")
for k, v in results.items():
    print(f"{k:<28} {v['val_pr_auc']:>8} {v['test_pr_auc']:>8}")
print("\nReference points on this same restricted sample set:")
print("  channel-stacked n=3 (notebook 07): test 0.3638")
print("  channel-stacked n=1 (notebook 07): test 0.4034")

**How to read Section 2**

- **ConvLSTM clears 0.4034** - recurrence recovers what channel stacking lost, and
  architecture was part of the problem. Worth carrying into fusion.
- **ConvLSTM lands near 0.36-0.40** - the constraint is the hourly interval, not the
  architecture. This is the stronger evidence for sub-hourly imagery, because it rules
  out the obvious alternative explanation.
- **B close to A** - the model was not leaning on timing, so the earlier result was
  about imagery rather than the clock.
- **B well below A** - a meaningful part of A's score came from time of day. That is
  worth stating plainly: it revises how much of the vision signal is genuinely visual.

## Section 3 - Stratified evaluation by time of day

An aggregate score can hide a real effect that exists only in convective hours. PR-AUC
is computed within four time-of-day bins so that the diurnal cycle is held roughly
constant inside each. If motion helps anywhere, the afternoon convective window is where
it should appear.

In [ ]:
from sklearn.metrics import average_precision_score
from datetime import datetime
import pandas as pd

def hour_of(key):
    return datetime.fromisoformat(key.split("|", 1)[1]).hour

BINS = [("night 00-05", range(0, 6)), ("morning 06-11", range(6, 12)),
        ("afternoon 12-17", range(12, 18)), ("evening 18-23", range(18, 24))]

hours = np.array([hour_of(k) for k in test_base.keys])
rows = []
for name, hrs in BINS:
    m = np.isin(hours, list(hrs))
    if m.sum() < 50 or ev_a["labels"][m].sum() < 5:
        rows.append({"window": name, "n": int(m.sum()), "positives": int(ev_a["labels"][m].sum()),
                     "base_rate": None, "convlstm": None, "time_controlled": None})
        continue
    rows.append({
        "window": name,
        "n": int(m.sum()),
        "positives": int(ev_a["labels"][m].sum()),
        "base_rate": round(float(ev_a["labels"][m].mean()), 4),
        "convlstm": round(float(average_precision_score(ev_a["labels"][m], ev_a["probs"][m])), 4),
        "time_controlled": round(float(average_precision_score(ev_b["labels"][m], ev_b["probs"][m])), 4),
    })

strat = pd.DataFrame(rows)
print(strat.to_string(index=False))
print("\nCompare each score against the base rate in the same row: that is the honest")
print("floor once time of day is held constant.")
results["stratified_by_time_of_day"] = rows

## Section 4 - Fusion with the ConvLSTM vision branch

The better of A and B is carried into late fusion against the tabular branch.

One point of care: this evaluation runs on the n = 3-eligible subset, not the full test
set, so Model 1's headline 0.7557 is not the right comparator here. The tabular branch is
rescored on exactly these hours and that number is the reference.

In [ ]:
import src.models.fusion as fus
importlib.reload(fus)

BEST = "convlstm_time_controlled" if (
    results["convlstm_time_controlled"]["test_pr_auc"] >
    results["convlstm_plain"]["test_pr_auc"]) else "convlstm_plain"
best_model = model_b if BEST == "convlstm_time_controlled" else model_a
best_val_ds  = val_t  if BEST == "convlstm_time_controlled" else val_base
best_test_ds = test_t if BEST == "convlstm_time_controlled" else test_base
print("carrying forward:", BEST)

def vision_probs(model, ds, keys):
    loader = DataLoader(ds, batch_size=128, shuffle=False, num_workers=2)
    ev = cl.evaluate(model, loader, dev)
    return pd.DataFrame({"key": keys, "p_img": ev["probs"], "y": ev["labels"]})

img_val  = vision_probs(best_model, best_val_ds,  val_base.keys)
img_test = vision_probs(best_model, best_test_ds, test_base.keys)

tab = fus.tabular_branch("models/xgboost_baseline.json",
                         "models/baseline_training.manifest.json")

def align(tab_frame, img_frame):
    m = tab_frame.merge(img_frame[["key", "p_img", "y"]], on="key", how="inner",
                        suffixes=("_tab", ""))
    ycol = "y" if "y" in m.columns else "y_tab"
    return {"y": m[ycol].to_numpy(), "p_tab": m["p_tab"].to_numpy(),
            "p_img": m["p_img"].to_numpy(), "n": len(m)}

val_al, test_al = align(tab["val"][0], img_val), align(tab["test"][0], img_test)
print(f"aligned val/test rows: {val_al['n']}/{test_al['n']}")

p_late, _ = fus.late_fusion(val_al, test_al)
tab_here = average_precision_score(test_al["y"], test_al["p_tab"])

print("\n==== Fusion on the n=3-eligible subset ====")
print(f"{'branch':<34} {'test PR-AUC':>12}")
print(f"{'Model 1 tabular (this subset)':<34} {tab_here:>12.4f}")
print(f"{'ConvLSTM vision':<34} {average_precision_score(test_al['y'], test_al['p_img']):>12.4f}")
print(f"{'Late fusion':<34} {average_precision_score(test_al['y'], p_late):>12.4f}")
print("\nModel 1 on the full test set is 0.7557; the subset figure above is the")
print("like-for-like comparator for these hours.")

results["fusion_on_subset"] = {
    "tabular_this_subset": round(float(tab_here), 4),
    "convlstm_vision": round(float(average_precision_score(test_al["y"], test_al["p_img"])), 4),
    "late_fusion": round(float(average_precision_score(test_al["y"], p_late)), 4),
    "n_rows": int(test_al["n"]), "vision_branch": BEST}

## Section 5 - Save results

In [ ]:
import json
from datetime import datetime, timezone

payload = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "experiment": "ConvLSTM vision branch and diurnal/semidiurnal control",
    "bands": BANDS,
    "n_frames": NF,
    "fairness": "all configs on the n=3-eligible common sample set (same as notebook 07)",
    "reference_channel_stacked": {"n1": 0.4034, "n2": 0.3809, "n3": 0.3638},
    "reference_model1_full_test": 0.7557,
    "notes": ("ConvLSTM trained from scratch; ResNet-18 baseline is ImageNet-pretrained, "
              "so the architectures are not compared on equal footing. Frame spacing "
              "remains hourly."),
    "results": results,
}
out = f"{DRIVE}/models/convlstm_results.json"
json.dump(payload, open(out, "w"), indent=2, default=str)
print("saved:", out)
print(json.dumps({k: v for k, v in results.items()
                  if k != "stratified_by_time_of_day"}, indent=2, default=str))